In [0]:
dbutils.widgets.text("config_path", "/Volumes/dbx_dev/stratus/eventhub_volume/config/customer_cdc_config.json")
config_path = dbutils.widgets.get("config_path")

import json
with open(config_path, "r") as f:
    config = json.load(f)
from pyspark.sql.functions import col, from_json, current_timestamp, when
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

# =========================================================
# Paths
# =========================================================
bronze_path = "/Volumes/dbx_dev/stratus/eventhub_volume/bronze/"
silver_path = "/Volumes/dbx_dev/stratus/eventhub_volume/silver/"
dlq_path = "/Volumes/dbx_dev/stratus/eventhub_volume/dlq/"

# =========================================================
# CDC event schema
# =========================================================
cdc_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("op", StringType(), True),
    StructField("entity", StringType(), True),
    StructField("record_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("country", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("source_ts", StringType(), True)
])

# =========================================================
# Read Bronze
# =========================================================
bronze_df = spark.read.format("delta").load(bronze_path)

# IMPORTANT:
# In Bronze you currently have the raw Event Hubs payload in `body`
# It looked base64-ish in the display, so first cast it to string.
bronze_payload_df = bronze_df.withColumn("body_str", col("body").cast("string"))

# =========================================================
# Parse JSON
# =========================================================
parsed_df = bronze_payload_df.withColumn(
    "parsed_json",
    from_json(col("body_str"), cdc_schema)
)

flattened_df = parsed_df.select(
    col("body_str"),
    col("partition"),
    col("offset"),
    col("sequenceNumber"),
    col("enqueuedTime"),
    col("publisher"),
    col("partitionKey"),
    col("properties"),
    col("parsed_json.event_id").alias("event_id"),
    col("parsed_json.op").alias("op"),
    col("parsed_json.entity").alias("entity"),
    col("parsed_json.record_id").alias("record_id"),
    col("parsed_json.name").alias("name"),
    col("parsed_json.email").alias("email"),
    col("parsed_json.country").alias("country"),
    col("parsed_json.updated_at").alias("updated_at"),
    col("parsed_json.source_ts").alias("source_ts")
)

# =========================================================
# Validation rules
# =========================================================
validated_df = flattened_df.withColumn(
    "is_valid",
    when(col("event_id").isNull(), False)
    .when(col("op").isNull(), False)
    .when(col("entity").isNull(), False)
    .when(col("record_id").isNull(), False)
    .otherwise(True)
).withColumn(
    "validation_error",
    when(col("event_id").isNull(), "Missing event_id")
    .when(col("op").isNull(), "Missing op")
    .when(col("entity").isNull(), "Missing entity")
    .when(col("record_id").isNull(), "Missing record_id")
    .otherwise(None)
).withColumn(
    "silver_processed_at",
    current_timestamp()
)

valid_df = validated_df.filter(col("is_valid") == True).drop("is_valid", "validation_error")
invalid_df = validated_df.filter(col("is_valid") == False)

# =========================================================
# Write Silver
# =========================================================
valid_df.write.format("delta").mode("overwrite").save(silver_path)

# =========================================================
# Write DLQ
# =========================================================
invalid_df.write.format("delta").mode("overwrite").save(dlq_path)

# =========================================================
# Show results
# =========================================================
print("Silver rows:", valid_df.count())
print("DLQ rows:", invalid_df.count())

display(valid_df)
display(invalid_df)

In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/dbx_dev/stratus/eventhub_volume/silver/customer/")
)

event_id,op,entity,source_ts,customer_id,customer_name,email,updated_at,json_str,partition,offset,event_timestamp,ingest_ts,entity_name,source_name
c12,D,customer,2026-04-13,101,null,null,null,"{""event_id"": ""c12"", ""op"": ""D"", ""entity"": ""customer"", ""customer_id"": 101, ""source_ts"": ""2026-04-13""}",1,23,2026-04-14T07:40:55.775Z,2026-04-14T07:41:04.206Z,customer,eventhub_customer


In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/silver/customer/", True)

False